## Overview
Notebook for selecting DS Slab Subduction sources to simulate

In [ ]:
import os
from pathlib import Path

import pandas as pd
import numpy as np
from plotly import graph_objects as go
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Markdown
import xarray as xr

import nn_gmm as nng
import ml_tools as mlt
import seismic_hazard_analysis as sha

In [ ]:
# Load the sites
# imdb_ffp = Path("/home/claudy/dev/work/data/nn_gmm/20251031_CS200m_rotd50_imdb.duckdb")
# imdb_ffp = Path("/Users/claudy/dev/work/data/nn_gmm/20251031_CS200m_rotd50_imdb.duckdb")

imdb_ffp = Path(os.environ["wdata"]) / "nn_gmm/20251031_CS200m_rotd50_imdb.duckdb"

with nng.DuckIMDB(imdb_ffp, readonly=True) as imdb:
    site_df = imdb.get_site_df(add_nztm=True, min_grid_level=0, max_grid_level=0)

In [ ]:
ds_source_df, ds_erf_df = nng.utils.get_ds_source_data()

### Source Magnitude Distribution by Tectonic Type

In [ ]:
mag_bins = ds_source_df.mag.unique() - 0.05
sns.displot(ds_source_df, x="mag", hue="tectonic_type", multiple="stack", bins=mag_bins, height=6, aspect=2)

### Source Magnitude Distribution By Depth

In [ ]:
unique_mag_values = np.unique(ds_source_df.mag)
mag_bins = np.linspace(start=unique_mag_values.min() - 0.05, stop=unique_mag_values.max() + 0.05, num=unique_mag_values.size + 1)
sns.displot(ds_source_df, x="mag", hue="depth", multiple="stack", bins=mag_bins, height=6, aspect=2)

### Source Depth Distribution By Tectonic Type

In [ ]:
unique_depth_values = np.unique(ds_source_df.depth)
depth_bins = np.linspace(start=unique_depth_values.min() - 10, stop=unique_depth_values.max() + 10, num=unique_depth_values.size + 1)
sns.displot(ds_source_df, x="depth", hue="tectonic_type", multiple="stack", bins=depth_bins, height=6, aspect=2)

## Spatial Map
Note: Everything from here is only considering slab sources, all crustal sources are ignored

In [ ]:
ds_source_df = ds_source_df.loc[ds_source_df.tectonic_type == "SUBDUCTION_SLAB"]

In [ ]:
source_loc_groups = ds_source_df.groupby("fault_name", observed=True)[["lon", "lat"]]
source_loc_df = source_loc_groups.first()

source_loc_df["count"] = source_loc_groups.size()

In [ ]:
sel_site_df = site_df.sort_values(["nztm_y", "nztm_x"]).iloc[::4, :]

In [ ]:
fig = go.Figure()
fig.update_layout(width=1600, height=800,
                  margin=dict(l=10, r=10, t=30, b=10),
                  map=dict(zoom=6, center=dict(lat=source_loc_df.lat.mean(), lon=source_loc_df.lon.mean())),
)

fig.add_trace(
    go.Scattermap(
        lon=source_loc_df["lon"],
        lat=source_loc_df["lat"],
        mode="markers",
        marker=dict(color="black", size=4),
        hovertemplate=(
            # "Lat: %{lat:.6f}<br>"
            # "Lon: %{lon:.6f}<br>"
            "Count: %{customdata[0]}<br>"
            "<extra></extra>"
        ),
        customdata=np.asarray(
            [
                source_loc_df["count"].values,
            ]
        ).T,
        name="DS Slab Source Locations",
    )
)

fig.add_trace(
    go.Scattermap(
        lon=site_df["lon"],
        lat=site_df["lat"],
        mode="markers",
        marker=dict(color="orange", size=4),
        hovertemplate=(
            "Site Int ID: %{customdata[0]}<br>"
            "<extra></extra>"
        ),
        customdata=np.asarray(
            [
                site_df.index.values,
            ]
        ).T,
        name="Level 0 Sites",
    )
)

fig.add_trace(
    go.Scattermap(
        lon=sel_site_df["lon"],
        lat=sel_site_df["lat"],
        mode="markers",
        marker=dict(color="red", size=6),
        hovertemplate=(
            "Site Int ID: %{customdata[0]}<br>"
            "<extra></extra>"
        ),
        customdata=np.asarray(
            [
                sel_site_df.index.values,
            ]
        ).T,
        name="Selected Sites",
    )
)

fig.show()

In [ ]:
# Save selected sites to file
#np.save("/Users/claudy/dev/work/data/nn_gmm/ds_simulations/disagg/sel_sites.npy", sel_site_df.site_id.values.astype(str))

## Disaggregation Results

In [ ]:
# disagg_results_dir = Path("/Users/claudy/dev/work/tmp/nn_gmm/disagg")
disagg_results_dir = Path(os.environ["wdata"]) / "nn_gmm/results/ds_simulations/disagg"
sites_disagg = {cur_ffp.stem: xr.load_dataarray(cur_ffp) for cur_ffp in disagg_results_dir.glob("*.nc")}

In [ ]:
depth_bin_labels = [f"{int(cur_bin)}_{int(cur_bin + 10)}" for cur_bin in depth_bins[:-1]]
mag_bin_labels = [f"{mag_bins[i]:.2f}_{mag_bins[i+1]:.2f}" for i in range(len(mag_bins) - 1)]

In [ ]:
sites = list(sites_disagg.keys())
ims = sites_disagg[sites[0]].coords["im_type"].values
rps = sites_disagg[sites[0]].coords["return_period"].values

In [ ]:
site_results = []
for site in sites:
    im_results = []
    
    cur_site_disagg = sites_disagg[site]

    for im in ims:
        cur_disagg = cur_site_disagg.sel(im_type=im).to_pandas()    
        cur_disagg["depth"] = ds_source_df.loc[cur_disagg.index, "depth"]
        cur_disagg["mag"] = ds_source_df.loc[cur_disagg.index, "mag"]

        cur_disagg["depth_bin"] = pd.cut(cur_disagg["depth"], bins=depth_bins, labels=depth_bin_labels)
        cur_disagg["mag_bin"] = pd.cut(cur_disagg["mag"], bins=mag_bins, labels=mag_bin_labels)

        cur_binned_disagg = cur_disagg.groupby(["depth_bin", "mag_bin"], as_index=False, observed=True)[rps].sum()
        cur_binned_disagg = xr.DataArray(data=np.stack([cur_binned_disagg[rp].values.reshape(len(depth_bin_labels), len(mag_bin_labels)) for rp in rps], axis=-1), 
                                    coords={"depth_bin": depth_bin_labels, "mag_bin": mag_bin_labels, "rp": rps}, dims=["depth_bin", "mag_bin", "rp"])
        
        im_results.append(cur_binned_disagg)

    cur_site_disagg = xr.concat(im_results, dim="im").assign_coords(im=ims)
    site_results.append(cur_site_disagg)

binned_disagg = xr.concat(site_results, dim="site").assign_coords(site=sites)

In [ ]:
depth_bins_centers = (depth_bins[:-1] + depth_bins[1:]) / 2 
mag_bin_centres = (mag_bins[:-1] + mag_bins[1:]) / 2

In [ ]:
def create_mag_depth_disagg_plot(rp: int, ims: list[str], binned_disagg: xr.DataArray):
    for im in ims:
        cur_bin_disagg = binned_disagg.sel(im=im, rp=rp)
        cur_bin_disagg = cur_bin_disagg.mean(dim="site").to_pandas()

        create_single_mag_depth_disagg_plot(im, cur_bin_disagg)

def create_single_mag_depth_disagg_plot(im: str, cur_bin_disagg: pd.DataFrame, vmin: float =0, vmax: float =15, count_label: str = "Contribution (%)", multiplier: float =100):
    fig, axs = plt.subplot_mosaic(
        [[".", "histx", "."], ["cbar", "main", "histy"]],
        width_ratios=(0.1, 4, 1),
        height_ratios=(1, 4),
        layout="constrained",
        figsize=(16,8),
    )
    ax = axs["main"]
    ax_histx = axs["histx"]
    ax_histy = axs["histy"]
    ax_cbar = axs["cbar"]   

    r = ax.pcolormesh(mag_bin_centres, depth_bins_centers, cur_bin_disagg.values * multiplier, cmap="Blues", vmin=vmin, vmax=vmax)
    ax.set_xlabel("Magnitude")
    ax.set_ylabel("Depth (km)")
    ax.invert_yaxis()


    ax_histx.bar(mag_bin_centres, cur_bin_disagg.sum(axis=0).values * multiplier, width=np.diff(mag_bins), align="center", edgecolor="black")
    ax_histx.set_ylabel(count_label)
    ax_histx.set_xticklabels([])
    ax_histx.set_xlim(ax.get_xlim())

    ax_histy.barh(depth_bins_centers, cur_bin_disagg.sum(axis=1).values * multiplier, height=np.diff(depth_bins), align="center", edgecolor="black")
    ax_histy.set_xlabel(count_label)
    ax_histy.set_yticklabels([])
    ax_histy.set_ylim(ax.get_ylim())

    plt.colorbar(r, cax=ax_cbar, label=count_label, location="left")
    fig.text(0.95, 0.9, nng.utils.get_nice_im_name(im), ha="right", va="center", fontsize=16, fontweight="bold")

In [ ]:
display(Markdown(f"## Magnitude-Depth Disaggregation Across {len(sites)} Sites"))


### RP = 2500

In [ ]:
create_mag_depth_disagg_plot(rp=2500, ims=ims, binned_disagg=binned_disagg)

## Aggregate Disagg Results
To create single distribution, which is then used for sampling of sources

In [ ]:
binned_disagg.dims

In [ ]:
agg_disagg = binned_disagg.mean(dim=["site", "im"]).sel(rp=2500).to_pandas()

In [ ]:
create_single_mag_depth_disagg_plot("Agg", agg_disagg)

## Source Selection

### Location

In [ ]:
# Select locations
sel_source_locs_df = source_loc_df.iloc[::9, :]

In [ ]:
fig = go.Figure()
fig.update_layout(width=1600, height=800,
                  margin=dict(l=10, r=10, t=30, b=10),
                  map=dict(zoom=6, center=dict(lat=source_loc_df.lat.mean(), lon=source_loc_df.lon.mean())),
)

fig.add_trace(
    go.Scattermap(
        lon=sel_source_locs_df["lon"],
        lat=sel_source_locs_df["lat"],
        mode="markers",
        marker=dict(color="red", size=4),
        hovertemplate=(
            # "Lat: %{lat:.6f}<br>"
            # "Lon: %{lon:.6f}<br>"
            "Count: %{customdata[0]}<br>"
            "<extra></extra>"
        ),
        customdata=np.asarray(
            [
                sel_source_locs_df["count"].values,
            ]
        ).T,
        name="Selected Source Locations",
    )
)


fig.show()

### Ruptures

In [ ]:
# Events at the selected locations
valid_ruptures_df = ds_source_df.loc[ds_source_df.fault_name.isin(sel_source_locs_df.index)].copy()

# Add weight based on aggregated disagg results
valid_ruptures_df.loc[:, "depth_bin"] = pd.cut(valid_ruptures_df["depth"], bins=depth_bins, labels=depth_bin_labels)
valid_ruptures_df.loc[:, "mag_bin"] = pd.cut(valid_ruptures_df["mag"], bins=mag_bins, labels=mag_bin_labels)
depth_ind = agg_disagg.index.get_indexer(valid_ruptures_df["depth_bin"])
mag_ind = agg_disagg.columns.get_indexer(valid_ruptures_df["mag_bin"])
valid_ruptures_df["weight"] = agg_disagg.values[depth_ind, mag_ind]

In [ ]:
# Select one rupture per location 
sel_rupture_ids = []
for cur_loc in sel_source_locs_df.index:
    cur_ruptures = valid_ruptures_df.loc[valid_ruptures_df.fault_name == cur_loc]
    norm_weights = cur_ruptures["weight"] / cur_ruptures["weight"].sum()
    sel_rupture_ids.append(np.random.choice(cur_ruptures.index, size=1, p=norm_weights.values)[0])

In [ ]:
# Add ruptures in the corners
corner_masks = [
    (valid_ruptures_df["mag"] == valid_ruptures_df["mag"].min()) & (valid_ruptures_df["depth"] == valid_ruptures_df["depth"].min()),
    (valid_ruptures_df["mag"] == valid_ruptures_df["mag"].min()) & (valid_ruptures_df["depth"] == valid_ruptures_df["depth"].max()),
    (valid_ruptures_df["mag"] == valid_ruptures_df["mag"].max()) & (valid_ruptures_df["depth"] == valid_ruptures_df["depth"].min()),
    (valid_ruptures_df["mag"] == valid_ruptures_df["mag"].max()) & (valid_ruptures_df["depth"] == valid_ruptures_df["depth"].max()),    
]

corner_rupture_ids = []
for cur_mask in corner_masks:
    corner_ruptures = valid_ruptures_df.loc[cur_mask]

    if corner_ruptures.index.isin(sel_rupture_ids).sum() > 1:
        continue

    corner_rupture_ids.extend(corner_ruptures.sample(n=5).index.values.tolist())

print(f"Added {len(corner_rupture_ids)} corner ruptures.")    
sel_rupture_ids.extend(corner_rupture_ids)

In [ ]:
sel_ruptures_df = valid_ruptures_df.loc[sel_rupture_ids]

In [ ]:
sel_ruptures_dist = sel_ruptures_df.groupby(["depth_bin", "mag_bin"], observed=False, as_index=False).size().pivot_table(index="depth_bin", columns="mag_bin", values="size", aggfunc="sum", fill_value=0, observed=False)

In [ ]:
create_single_mag_depth_disagg_plot("Selected Ruptures", sel_ruptures_dist, vmin=0, vmax=45, count_label="Number of Ruptures", multiplier=1)

In [ ]:
# Save
# sel_ruptures_df.to_csv(disagg_results_dir / "selected_ruptures.csv")